# SOLiD-PINN v0.8.5 — Model mismatch reproduction
CPU runtime; Run all. No token, no private inputs, no GitHub writes. Reproduce all 24 fits using a pinned commit, test first, save a small synthetic metrics JSON to Drive and download it for review. Negative results are preserved.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess,sys,os,tempfile,json,hashlib
drive.mount("/content/drive")
REV="4ea0868db0c3d96a49384e9f58c2a86a923015d8"
REPO=Path(tempfile.mkdtemp(prefix="solid-pinn-v085-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
source=(REPO/"scripts/mismatch_benchmark.py").read_bytes()
assert hashlib.sha256(source).hexdigest()=="c924545b01db9c1d6a54c4429b9c533cfe4081ea64e55a638f98ce3ca78b7555"
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_mismatch_benchmark.py"],cwd=REPO,env=env,check=True)
out=REPO/"results.json"
subprocess.run([sys.executable,"scripts/mismatch_benchmark.py","--out",str(out)],cwd=REPO,env=env,check=True)
d=json.loads(out.read_text())
expected={(m,n,s) for m in ("misspecified","oracle","flexible","hybrid_pinn") for n in (0,1,5) for s in (11,23)}
assert len(d["metrics"])==24 and {(x["method"],x["noise_mv"],x["seed"]) for x in d["metrics"]}==expected
assert d["rk4_ground_truth_rmse"]<1e-6
d["source_revision"]=REV
d["source_sha256"]=hashlib.sha256(source).hexdigest()
d["targeted_tests_passed"]=4
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v085/handoff")
root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v085_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    part=dest.with_suffix(".part");part.write_bytes(raw);part.replace(dest)
assert dest.read_bytes()==raw
print("DRIVE VERIFIED",dest.name,sha)
files.download(str(dest))